# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library, referencing all entities (record sets, fields, columns) by their `@id`.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f'Dataset Name: {metadata.name}')
print(f'Dataset Description: {metadata.description}')

## 2. Data Overview

Review available record sets and fields. All entities are referenced by their `@id`.

We'll inspect the record sets present in this dataset, then look up the available fields for a chosen record set. The fields and columns will also be accessed and referenced by their `@id` values.


In [ ]:
# List record sets in the dataset and their @id
record_sets = list(dataset.metadata.record_sets)
print('Record sets and their @id:')
for rs in record_sets:
    print(f'- {rs.id}: {rs.name}')

# If no record sets found directly, try dataset.metadata.record_sets (newer mlcroissant versions)
if not record_sets:
    print('No record sets found directly in dataset metadata. Try loading from records:')
    record_set_ids = dataset.record_set_ids
    print('Available record sets by @id:')
    for rs_id in record_set_ids:
        print('-', rs_id)

In [ ]:
# Let's display the available record set ids as seen by the Dataset object
record_set_ids = dataset.record_set_ids
print('Available record set @id values:')
for rs_id in record_set_ids:
    print(rs_id)

# We'll use the first found record set for demonstration (replace with a specific known @id if desired)
record_set_id = record_set_ids[0]

In [ ]:
# Inspect available fields (by @id) in the selected record set
print(f'Fields in record set "{record_set_id}":')
fields = dataset.field_ids(record_set=record_set_id)
for field_id in fields:
    print('-', field_id)

In [ ]:
# Show sample records from the record set (referencing IDs)
print(f'Example record(s) from record set @id={record_set_id}:')
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    if i >= 3:
        break
    print(json.dumps(record, indent=2))

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. All extraction references the record set and field `@id`s from the overview.


In [ ]:
# Extract all record sets into dataframes, referencing @id
dataframes = {}

for rs_id in record_set_ids:
    print(f'Loading record set: {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f'  Columns: {list(df.columns)}  Rows: {len(df)}')

# Show sample of the first dataframe
first_df = dataframes[record_set_id]
print(f'Columns in first record set ({record_set_id}):')
print(first_df.columns.tolist())
first_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, referencing fields by their `@id`. We'll select a numeric field, filter, normalize, and (if possible) group.

_Note: Replace field/column @id with one actually present in the above extracted DataFrame. As an example, we'll use the first numeric-looking field._

In [ ]:
# Heuristically choose a numeric field (fallback: pick first column with int/float-like values)
import numpy as np

candidate_numeric_field = None
for col in first_df.columns:
    vals = first_df[col].dropna()
    if len(vals) == 0:
        continue
    # Try to coerce type to float
    try:
        vals_float = pd.to_numeric(vals)
        # If more than half of values are valid floats, treat as numeric
        if (vals_float.notnull().sum() / len(vals) > 0.8):
            candidate_numeric_field = col
            break
    except Exception:
        continue
if not candidate_numeric_field:
    candidate_numeric_field = first_df.columns[0]  # just fallback to first column

numeric_field_id = candidate_numeric_field
print(f'Using numeric field @id: {numeric_field_id}')

# Attempt to use a plausible group/id field for grouping (e.g., 'sex', 'gender', or similar)
potential_groups = ['sex', 'gender', 'tumor_site', 'histology', 'mmr_status', 'msi_status']
group_field = None
for g in potential_groups:
    for col in first_df.columns:
        if g in col.lower():
            group_field = col
            print(f'Using group field @id: {group_field}')
            break
    if group_field:
        break

# Convert numeric field to float
x = pd.to_numeric(first_df[numeric_field_id], errors='coerce')
# Filter for records with value > threshold (use the 10th percentile if min value is large)
threshold = 10
if x.min() > threshold:
    threshold = float(x.quantile(0.1))

filtered_df = first_df[x > threshold].copy()
print(f'Filtered records with {numeric_field_id} > {threshold}:')
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f'{numeric_field_id}_normalized'] = (x[x > threshold] - x[x > threshold].mean()) / x[x > threshold].std()
print(f'Normalized {numeric_field_id} for filtered records:')
print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Group by group_field if available
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f'Grouped data by {group_field} (mean {numeric_field_id}):')
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution and relation of processed fields in the dataset. All axes and labels reference the chosen field @id values.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8,4))

# Histogram of the selected numeric field
sns.histplot(pd.to_numeric(first_df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group_field is available, make a boxplot
if group_field and group_field in first_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=first_df, x=group_field, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{numeric_field_id} by {group_field}')
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to explore the FAIR^2 dataset of clinicopathological and molecular characteristics for second primary colorectal cancer in cancer survivors. By addressing fields and record sets via their `@id`, we:

- Loaded metadata and listed record sets and fields.
- Extracted tabular data into DataFrames.
- Applied basic EDA, such as filtering and normalizing a numeric field.
- Visualized distributions and group differences.

This demonstrates a reproducible workflow for referencing and exploring Croissant-encoded datasets. Further analyses can use the same pattern, always referencing entities by `@id` for robust and future-proof data science.